# MeetingMind — Colab notebook
**Your meetings, made meaningful.**

This notebook runs the *same* NLP pipeline as the MeetingMind web app (the `backend/app` Python package):
summaries, discussion topics, decisions, action items with owners and deadlines, risks and blockers,
unresolved questions, named entities, keywords and recommended next steps — each backed by transcript evidence.

**How to use it**
1. *Runtime → Change runtime type → GPU* is optional (CPU works; GPU makes summarization faster).
2. Run **Step 1** and upload `MeetingMind.zip` (the project archive) when asked — or set `REPO_URL` to a Git repository that contains the project.
3. Run Steps 2–3 to install dependencies and load the models (first run downloads ~1.3 GB).
4. Analyse the sample transcript, your own text, or a TXT upload (Steps 4–6), or open the Gradio interface (Step 7).

**Privacy:** everything runs inside this Colab runtime. Nothing is stored after the runtime is reset, except files you download yourself.
The React website remains the primary application; this notebook is for quick experiments and demonstrations.

## Step 1 — Get the MeetingMind code

In [ ]:
import os, sys, zipfile, glob, subprocess

REPO_URL = ""          # optional: e.g. "https://github.com/<you>/MeetingMind.git"
PROJECT_DIR = "/content/MeetingMind" if os.path.isdir("/content") else os.path.abspath("..")

def find_backend(root):
    hits = glob.glob(os.path.join(root, "**", "backend", "app", "nlp", "pipeline.py"), recursive=True)
    return os.path.dirname(os.path.dirname(os.path.dirname(hits[0]))) if hits else None

BACKEND_DIR = find_backend(PROJECT_DIR) or find_backend(os.getcwd())
if BACKEND_DIR is None and REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, PROJECT_DIR], check=True)
    BACKEND_DIR = find_backend(PROJECT_DIR)
if BACKEND_DIR is None:
    from google.colab import files  # only available on Colab
    print("Upload MeetingMind.zip (the project archive)…")
    uploaded = files.upload()
    for name in uploaded:
        if name.endswith(".zip"):
            with zipfile.ZipFile(name) as zf:
                zf.extractall(PROJECT_DIR)
    BACKEND_DIR = find_backend(PROJECT_DIR)

assert BACKEND_DIR, "Could not find backend/app — check the upload or REPO_URL."
sys.path.insert(0, BACKEND_DIR)
print("Using backend at", BACKEND_DIR)

## Step 2 — Install dependencies
Colab already ships PyTorch, so pip keeps the existing build. This takes 1–3 minutes.

In [ ]:
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q -r {BACKEND_DIR}/requirements.txt gradio
    !python -m spacy download en_core_web_sm -q
else:
    print("Not on Colab: assuming the project's virtual environment is already set up.")

## Step 3 — Load the NLP models
Models are loaded once and cached. Set `SUMMARY_MODE = "extractive"` to skip the 1.2 GB summarization model
(e.g. on a slow connection); everything else still works.

In [ ]:
import os, time
SUMMARY_MODE = "abstractive"     # or "extractive"
os.environ["MEETINGMIND_SUMMARY_MODE"] = SUMMARY_MODE
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

from app.config import get_settings
from app.nlp.models import ModelRegistry, set_registry
from app.nlp.pipeline import analyze
from app.schemas import AnalyzeRequest
from app.services import exports
from app.services.file_extraction import extract_text

get_settings.cache_clear()
registry = ModelRegistry(get_settings())
set_registry(registry)
for key in registry.keys():
    if registry.state(key) == "disabled":
        continue
    t = time.time()
    model = registry.try_get(key)
    print(f"{key:<11} {registry.name(key):<45} {'ready' if model is not None else 'FAILED: ' + str(registry.error(key))} ({time.time() - t:.1f}s)")

## Step 4 — Helper functions for displaying a report

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

def show_report(report):
    s = report.stats
    display(Markdown(f"# {report.meeting.title}\n"
                     f"{report.meeting.date or 'no date'} · {s.word_count} words · {s.speaker_count} speakers · "
                     f"summary method: **{report.summary.method}**"))
    display(Markdown("## Executive summary\n" + "\n".join(
        f"- {x.text}" + (" _(weakly supported)_" if x.grounding == "weak" else "") for x in report.summary.executive)))
    if report.action_items:
        display(Markdown("## Action items"))
        display(pd.DataFrame([{
            "id": a.id, "task": a.description, "owner": a.owner,
            "deadline": (a.deadline.normalized_date.isoformat() if a.deadline and a.deadline.normalized_date else (a.deadline.text if a.deadline else "")),
            "review": "; ".join(a.review_reasons), "evidence": a.evidence[0].text if a.evidence else "",
        } for a in report.action_items]))
    for title, rows in [
        ("Decisions", [{"status": d.status, "decision": d.text, "speaker": d.speaker} for d in report.decisions]),
        ("Risks and blockers", [{"kind": r.kind, "severity": r.severity, "text": r.text} for r in report.risks]),
        ("Unresolved questions", [{"status": q.status, "question": q.text, "asked by": q.asked_by} for q in report.questions]),
        ("Topics", [{"topic": t.title, "keywords": ", ".join(t.keywords), "sentences": len(t.sentence_ids)} for t in report.topics]),
        ("Recommended next steps", [{"priority": n.priority, "step": n.text, "why": n.rationale} for n in report.next_steps]),
    ]:
        if rows:
            display(Markdown(f"## {title}"))
            display(pd.DataFrame(rows))
    display(Markdown("**Keywords:** " + ", ".join(k.phrase for k in report.keywords)))
    if report.warnings:
        display(Markdown("**Notes:** " + " ".join(report.warnings)))

def save_exports(report, folder="meetingmind_exports"):
    os.makedirs(folder, exist_ok=True)
    paths = []
    for fmt, (fn, _media, ext) in exports.EXPORTERS.items():
        path = os.path.join(folder, exports.safe_filename(report.meeting.title, ext))
        with open(path, "wb") as fh:
            fh.write(fn(report))
        paths.append(path)
    return paths

## Step 5 — Analyse the sample transcript

In [ ]:
from datetime import date
sample_path = os.path.join(os.path.dirname(BACKEND_DIR), "samples", "sample_transcript.txt")
SAMPLE = open(sample_path, encoding="utf-8").read()
print(SAMPLE[:600], "…")

t = time.time()
report = analyze(AnalyzeRequest(transcript=SAMPLE, title="Mobile App Launch Sync", meeting_date=date(2026, 3, 4)))
print(f"Analysed in {time.time() - t:.1f}s")
show_report(report)

## Step 6 — Your own transcript (text or TXT upload)
Paste a transcript into `MY_TRANSCRIPT` (one line per speaker turn, e.g. `Priya: I'll send it Friday.`),
or set `UPLOAD = True` to upload a `.txt` file. Set `MEETING_DATE` so relative deadlines resolve correctly.

In [ ]:
MY_TRANSCRIPT = """
Ann: Thanks for joining. First, the website relaunch.
Ben: The new design is ready, but the hosting contract expires on April 30, which is a risk.
Ann: Ben, can you renew the hosting contract by next Friday?
Ben: Sure, I'll take care of it.
Cara: We decided to keep the old blog URLs so we don't lose search traffic.
Ann: Who is going to write the launch announcement?
Cara: I'm not sure yet, I need to check with marketing.
"""
MEETING_TITLE = "Website relaunch"
MEETING_DATE = date(2026, 4, 8)
UPLOAD = False

text = MY_TRANSCRIPT
if UPLOAD:
    from google.colab import files
    up = files.upload()
    name, data = next(iter(up.items()))
    text = extract_text(name, data, max_bytes=10 * 1024 * 1024).text   # TXT, DOCX or PDF

my_report = analyze(AnalyzeRequest(transcript=text, title=MEETING_TITLE, meeting_date=MEETING_DATE))
show_report(my_report)
paths = save_exports(my_report)
print("Saved:", *paths, sep="\n  ")
if "google.colab" in sys.modules:
    from google.colab import files
    for p in paths:
        files.download(p)

## Step 7 — Gradio interface
A simple web UI inside Colab. With `share=True` Gradio creates a temporary public link (anyone with the link can use it
while this runtime is running); use `share=False` to keep it inside the notebook.

In [ ]:
import gradio as gr
from datetime import datetime

def run_analysis(transcript, upload, title, meeting_date):
    if upload is not None:
        path = upload if isinstance(upload, str) else upload.name
        with open(path, "rb") as fh:
            transcript = extract_text(os.path.basename(path), fh.read(), max_bytes=10 * 1024 * 1024).text
    if not transcript or not transcript.strip():
        raise gr.Error("Paste a transcript or upload a file.")
    md_date = None
    if meeting_date:
        try:
            md_date = datetime.strptime(str(meeting_date).strip()[:10], "%Y-%m-%d").date()
        except ValueError:
            raise gr.Error("Meeting date must look like 2026-03-04.")
    try:
        rep = analyze(AnalyzeRequest(transcript=transcript, title=title or "Untitled meeting", meeting_date=md_date))
    except ValueError as exc:   # validation errors (too short, too long, …)
        raise gr.Error(str(exc))
    actions = pd.DataFrame([{
        "task": a.description, "owner": a.owner,
        "deadline": a.deadline.normalized_date.isoformat() if a.deadline and a.deadline.normalized_date else (a.deadline.text if a.deadline else ""),
        "needs review": "yes" if a.needs_review else "",
    } for a in rep.action_items]) if rep.action_items else pd.DataFrame(columns=["task", "owner", "deadline", "needs review"])
    return exports.to_markdown(rep), actions, save_exports(rep, folder=f"exports_{rep.report_id}")

with gr.Blocks(title="MeetingMind", theme=gr.themes.Soft(primary_hue="violet", secondary_hue="pink")) as demo:
    gr.Markdown("# MeetingMind\n*Your meetings, made meaningful.*")
    with gr.Row():
        with gr.Column(scale=1):
            title_in = gr.Textbox(label="Meeting title", value="Mobile App Launch Sync")
            date_in = gr.Textbox(label="Meeting date (YYYY-MM-DD)", value="2026-03-04")
            file_in = gr.File(label="Or upload TXT / DOCX / PDF", file_types=[".txt", ".docx", ".pdf", ".vtt", ".srt"])
            go = gr.Button("Analyze meeting", variant="primary")
        with gr.Column(scale=2):
            text_in = gr.Textbox(label="Transcript", lines=16, value=SAMPLE)
    with gr.Tab("Report"):
        report_out = gr.Markdown()
    with gr.Tab("Action items"):
        actions_out = gr.Dataframe()
    with gr.Tab("Downloads"):
        files_out = gr.File(label="JSON · CSV · Markdown · PDF", file_count="multiple")
    go.click(run_analysis, [text_in, file_in, title_in, date_in], [report_out, actions_out, files_out])

if "google.colab" in sys.modules:
    demo.launch(share=True, debug=False)
else:
    print("Gradio interface built; call demo.launch() to start it locally.")

## Notes and limitations
- Extraction combines linguistic rules (spaCy) with sentence-embedding similarity; it is strongest on transcripts with
  `Name: text` speaker labels. Items that were inferred or are ambiguous are flagged for review.
- The abstractive summary is checked against the transcript (unsupported names/numbers are removed and conflated facts
  are flagged), but it can still be imperfect — always check the evidence.
- The "signal" labels are heuristic strengths, **not** calibrated probabilities.
- See `README.md` and `docs/` in the project for the architecture, evaluation results and viva notes.